# 09 - 插件系统与 Hook 生命周期

> **何时使用**: 当你需要扩展 sqlseed 的功能（如自动添加时间戳、数据脱敏、自定义生成器）时。
>
> **核心概念**: pluggy 插件框架，11 个 Hook 覆盖从注册到写入的完整生命周期。

## 适用场景

- 每行数据自动添加字段（如 `created_at`）→ `transform_row` Hook
- 批量数据变换（如全部大写）→ `transform_batch` Hook
- 自定义数据生成器 → `register_providers` Hook
- 写入前后执行逻辑 → `before_insert` / `after_insert` Hook

## 你将学到

- 11 个 Hook 生命周期
- 自定义 Provider 开发
- PluginMediator 桥接机制
- entry-point 打包流程

详见 architecture.zh-CN.md §8

**📚 教程导航**

| 序号 | 主题 | 架构层 | 前置要求 |
|------|------|--------|----------|
| 01 | 快速上手与核心流程 | Orchestrator | 无 |
| 02 | 9 级策略链详解 | Core: ColumnMapper | 01 |
| 03 | 生成器与 Provider 体系 | Generators | 01 |
| 04 | 数据库层与多表关联 | Database + Core | 01 |
| 05 | 表达式派生与约束求解 | Core: DAG / Expression | 01 |
| 06 | 配置驱动与 Transform | Config / Core | 01 |
| 07 | AI 智能配置 | Plugins: AI | 01 |
| 08 | MCP 服务器集成 | Plugins: MCP | 07 |
| **→ 09** | **插件系统与 Hook 生命周期** | **Plugins** | **01** |
| 10 | CLI 参考手册 | CLI | 06 |
| 11 | 工具类参考 | Utils | 01 |
| 12 | 测试集成模式 | Testing | 01 |

---

In [1]:
from sqlseed.generators._dispatch import GeneratorDispatchMixin
from sqlseed.generators._protocol import DataProvider
from sqlseed.plugins.hookspecs import hookimpl

import sqlite3
# Prerequisite: pip install -e ".[dev,all]"
import sqlseed
from sqlseed import connect, fill, fill_from_config, preview

# Demo database setup
import sys; sys.path.insert(0, "..")  # for build_demo_db only
from build_demo_db import build
db_path = build()  # Force rebuild to ensure idempotent run

# Populate base dependencies
with connect(str(db_path)) as orch:
    orch.fill_table("organizations", count=5, seed=42)
    orch.fill_table("members", count=20, seed=42)
    orch.fill_table("projects", count=10, seed=42)
    orch.fill_table("tags", count=8, seed=42)

print(f"sqlseed {sqlseed.__version__} | Database: {db_path}")

Generating organizations:   0%|          | 0/5 [00:00<?, ?it/s]

Generating members:   0%|          | 0/20 [00:00<?, ?it/s]

Generating projects:   0%|          | 0/10 [00:00<?, ?it/s]

Generating tags:   0%|          | 0/8 [00:00<?, ?it/s]

sqlseed 0.1.16.dev1+g0824e8553.d20260505 | Database: /Users/sunbo/Documents/webblock/sqlseed/examples/sqlseed_demo.db


### 📍 架构定位

| 模块 | 文件 | 核心类/函数 |
|------|------|------------|
| 插件 Hook 规范 | `src/sqlseed/plugins/hookspecs.py` | `SqlseedHookSpec` |

> 对应架构图: [§8 插件 Hook 生命周期](../docs/architecture.zh-CN.md#8-插件-hook-生命周期)

## 1. 先看效果 — 插件自动添加时间戳

注册一个插件，在每行数据写入前自动添加 `created_at` 时间戳 — **零侵入**：

```python
class TimestampPlugin:
    @hookimpl
    def sqlseed_transform_row(self, table_name, row):
        row['created_at'] = datetime.now().isoformat()
        return row
```

下面演示插件开发和 11 个 Hook 生命周期。

## 2. 自定义 Provider 开发

自定义 Provider 需要实现 `DataProvider` Protocol 并继承 `GeneratorDispatchMixin`：

In [2]:
import secrets
import typing


class ChineseNameProvider(GeneratorDispatchMixin, DataProvider):
    name = "chinese_name"
    _GENERATOR_MAP: typing.ClassVar[dict] = {}

    def __init__(self):
        self._locale = "zh"
        self._seed = None
        self._last_names = ["张", "王", "李", "赵", "刘", "陈", "杨", "黄", "周", "吴"]
        self._first_names = ["伟", "芳", "秀英", "敏", "静", "丽", "强", "磊", "洋", "勇"]

    def set_locale(self, locale: str) -> None:
        self._locale = locale

    def set_seed(self, seed: int | None) -> None:
        self._seed = seed

    def generate(self, type_name: str, **params: object) -> object:
        last = secrets.choice(self._last_names)
        first = secrets.choice(self._first_names)
        return f"{last}{first}"


provider = ChineseNameProvider()
print(f"Provider: {provider.name}")
print(f"Sample: {provider.generate('name')}")
print(f"Sample: {provider.generate('name')}")
print(f"Sample: {provider.generate('name')}")

Provider: chinese_name
Sample: 赵磊
Sample: 黄伟
Sample: 刘磊


## 3. 11 个 Hook 生命周期

sqlseed 定义了 11 个 Hook，按执行阶段分为：

| 阶段 | Hook | 说明 |
|---|---|---|
| 注册 | sqlseed_register_providers | 注册自定义 Provider |
| 注册 | sqlseed_register_column_mappers | 注册自定义列映射规则 |
| AI分析 | sqlseed_ai_analyze_table | AI 分析表结构 (firstresult) |
| AI分析 | sqlseed_pre_generate_templates | 预生成模板值 (firstresult) |
| 生成 | sqlseed_before_generate | 生成前回调 |
| 生成 | sqlseed_after_generate | 生成后回调 |
| 生成 | sqlseed_transform_row | 逐行变换 (热路径) |
| 生成 | sqlseed_transform_batch | 批量变换 (链式) |
| 写入 | sqlseed_before_insert | 插入前回调 |
| 写入 | sqlseed_after_insert | 插入后回调 |
| 共享池 | sqlseed_shared_pool_loaded | 共享池加载完成 |

In [3]:
hookspec_names = [
    ("sqlseed_register_providers", "注册阶段", "注册自定义 Provider"),
    ("sqlseed_register_column_mappers", "注册阶段", "注册自定义列映射规则"),
    ("sqlseed_ai_analyze_table", "AI分析阶段", "AI 分析表结构 (firstresult)"),
    ("sqlseed_pre_generate_templates", "AI分析阶段", "预生成模板值 (firstresult)"),
    ("sqlseed_before_generate", "生成阶段", "生成前回调"),
    ("sqlseed_after_generate", "生成阶段", "生成后回调"),
    ("sqlseed_transform_row", "生成阶段", "逐行变换 (热路径)"),
    ("sqlseed_transform_batch", "生成阶段", "批量变换 (链式)"),
    ("sqlseed_before_insert", "写入阶段", "插入前回调"),
    ("sqlseed_after_insert", "写入阶段", "插入后回调"),
    ("sqlseed_shared_pool_loaded", "共享池阶段", "共享池加载完成"),
]

print("sqlseed 11 个 Hook 生命周期:\n")
for i, (name, phase, desc) in enumerate(hookspec_names, 1):
    print(f"  {i:2d}. [{phase}] {name}")
    print(f"      {desc}")

sqlseed 11 个 Hook 生命周期:

   1. [注册阶段] sqlseed_register_providers
      注册自定义 Provider
   2. [注册阶段] sqlseed_register_column_mappers
      注册自定义列映射规则
   3. [AI分析阶段] sqlseed_ai_analyze_table
      AI 分析表结构 (firstresult)
   4. [AI分析阶段] sqlseed_pre_generate_templates
      预生成模板值 (firstresult)
   5. [生成阶段] sqlseed_before_generate
      生成前回调
   6. [生成阶段] sqlseed_after_generate
      生成后回调
   7. [生成阶段] sqlseed_transform_row
      逐行变换 (热路径)
   8. [生成阶段] sqlseed_transform_batch
      批量变换 (链式)
   9. [写入阶段] sqlseed_before_insert
      插入前回调
  10. [写入阶段] sqlseed_after_insert
      插入后回调
  11. [共享池阶段] sqlseed_shared_pool_loaded
      共享池加载完成


## 4. 实战：transform_row Hook

注册一个插件，在每行数据写入前自动添加 `created_at` 时间戳。

In [4]:
from datetime import datetime

import pluggy


class TimestampPlugin:
    """Adds created_at timestamp to each row."""

    @hookimpl
    def sqlseed_transform_row(self, table_name: str, row: dict) -> dict | None:
        if 'created_at' not in row or row['created_at'] is None:
            row['created_at'] = datetime.now().isoformat()
            return row
        return None

# Register and use with fill

with connect(str(db_path), provider='mimesis', locale='en') as orch:
    orch._ext.plugins.register(TimestampPlugin())
    result = orch.fill_table('organizations', count=5, clear_before=True, seed=42,
        columns={"org_code": {"type": "pattern", "regex": ORG_PATTERN},
                 "parent_code": {"type": "choice", "choices": [*[r[0] for r in __import__("sqlite3").connect(str(db_path)).execute("SELECT org_code FROM organizations").fetchall()]]}})

print(f'Filled {result.count} rows in {result.elapsed:.3f}s')
conn = sqlite3.connect(str(db_path))
rows = conn.execute('SELECT org_code, name, created_at FROM organizations').fetchall()
for row in rows:
    print(f'  {row[0]}: {row[1]} | created_at={row[2]}')
conn.close()


Generating organizations:   0%|          | 0/5 [00:00<?, ?it/s]

Filled 5 rows in 0.034s
  ORG-0433: Anthony Reilly | created_at=2020-02-01 23:17:15.234053
  ORG-1819: Kai Day | created_at=2004-12-04 21:47:57.571858
  ORG-0013: Cleveland Osborn | created_at=2002-10-14 01:01:05.229258
  ORG-8908: Zack Holder | created_at=2007-09-20 00:35:12.750800
  ORG-8637: Arden Brady | created_at=2020-12-18 13:14:28.617889


## 5. transform_batch Hook

`transform_batch` 对整批数据进行变换，适合批量计算或过滤。

In [5]:
class UppercasePlugin:
    """Uppercase all string values in each batch."""

    @hookimpl
    def sqlseed_transform_batch(self, table_name: str, batch: list[dict]) -> list[dict] | None:
        for row in batch:
            for key, val in row.items():
                if isinstance(val, str) and key not in ('created_at', 'registered_at', 'due_at', 'completed_at', 'uploaded_at'):  # noqa: E501
                    row[key] = val.upper()
        return batch

with connect(str(db_path), provider='mimesis', locale='en') as orch:
    orch._ext.plugins.register(UppercasePlugin())
    result = orch.fill_table('organizations', count=3, clear_before=True, seed=42,
        columns={"org_code": {"type": "pattern", "regex": ORG_PATTERN},
                 "parent_code": {"type": "choice", "choices": [*[r[0] for r in __import__("sqlite3").connect(str(db_path)).execute("SELECT org_code FROM organizations").fetchall()]]}})
    print(f'Filled {result.count} rows with UppercasePlugin')

conn = sqlite3.connect(str(db_path))
rows = conn.execute('SELECT org_code, name FROM organizations').fetchall()
for row in rows:
    print(f'  {row[0]}: {row[1]}')
conn.close()


Generating organizations:   0%|          | 0/3 [00:00<?, ?it/s]

Filled 3 rows with UppercasePlugin
  ORG-0433: ANTHONY REILLY
  ORG-1819: KAI DAY
  ORG-0013: CLEVELAND OSBORN


## 6. 打包为独立插件

通过 `entry-point` 机制，sqlseed 自动发现并加载已安装的插件包。

In [6]:
# Minimal pyproject.toml for a sqlseed plugin
print('''[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "sqlseed-my-plugin"
version = "0.1.0"
dependencies = ["sqlseed>=0.1.0"]

[project.entry-points.sqlseed]
my_plugin = "my_plugin.plugin"''')

print('\nPlugin module (my_plugin/plugin.py):')
print('''from sqlseed.plugins.hookspecs import hookimpl

class MyPlugin:
    @hookimpl
    def sqlseed_transform_row(self, table_name, row):
        row["custom_field"] = "processed"
        return row''')

[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "sqlseed-my-plugin"
version = "0.1.0"
dependencies = ["sqlseed>=0.1.0"]

[project.entry-points.sqlseed]
my_plugin = "my_plugin.plugin"

Plugin module (my_plugin/plugin.py):
from sqlseed.plugins.hookspecs import hookimpl

class MyPlugin:
    @hookimpl
    def sqlseed_transform_row(self, table_name, row):
        row["custom_field"] = "processed"
        return row


## 总结

| Hook | 阶段 | 用途 |
|------|------|------|
| `sqlseed_register_providers` | 注册 | 自定义 Provider |
| `sqlseed_register_column_mappers` | 注册 | 自定义列映射 |
| `sqlseed_before_generate` | 生成前 | 准备工作 |
| `sqlseed_transform_row` | 生成中 | 逐行变换 |
| `sqlseed_transform_batch` | 生成中 | 批量变换 |
| `sqlseed_after_generate` | 生成后 | 收尾工作 |
| `sqlseed_before_insert` | 写入前 | 预处理 |
| `sqlseed_after_insert` | 写入后 | 记录统计 |

**下一步**: [10-cli-reference.ipynb](10-cli-reference.ipynb) — CLI 参考手册

In [7]:
# ✅ 验证: 确保数据已成功生成并写入
import sqlite3
conn = sqlite3.connect(str(db_path))
try:
    # 基本行数验证
    member_count = conn.execute("SELECT COUNT(*) FROM members").fetchone()[0]
    assert member_count > 0, f"Expected members > 0, got {member_count}"
    print("✅ All assertions passed")
finally:
    conn.close()

✅ All assertions passed
